# Module 00 Lab — Build a Bare-Metal Agent

**Course:** AI Agents Teaching Platform  
**Module:** 00 — Mental Models  
**Estimated time:** 2–3 hours

---

### What you'll build

A working agent loop **from scratch** — no LangChain, no agent frameworks, no SDK abstractions beyond the API client. Just Python.

By the end you'll have:
- A safe calculator tool
- A ~50-line agent loop that calls the LLM, executes tools, and terminates correctly

### Constraints
- No LangChain, LangGraph, AutoGen, or similar frameworks
- Raw API calls only
- Loop must have a working termination condition **and** a `MAX_STEPS` guardrail

---

> **VS Code / local?** You can run this lab locally too — see the lab page on the platform for instructions.

## Step 1 — Choose your provider

Set `PROVIDER` to `"anthropic"` or `"openai"`. Everything else adapts automatically.

| Provider | Model used | Free tier? |
|---|---|---|
| `anthropic` | `claude-haiku-4-5-20251001` | No — pay-as-you-go, very cheap |
| `openai` | `gpt-4o-mini` | No — pay-as-you-go, very cheap |

Both cost fractions of a cent per run. Get a key at [console.anthropic.com](https://console.anthropic.com) or [platform.openai.com](https://platform.openai.com).

---

> **Why does the provider not matter for the loop?**
> This is the point of the lab. The agent loop — the `for` loop, the tool registry, the message list, the termination check — is **identical** for both providers. Only `call_model` changes. The LLM is one swappable component; everything else is yours.

In [ ]:
# ── Pick one ──────────────────────────────────────────────
PROVIDER = "anthropic"   # "anthropic" or "openai"
# ──────────────────────────────────────────────────────────

MODELS = {
    "anthropic": "claude-haiku-4-5-20251001",
    "openai":    "gpt-4o-mini",
}

assert PROVIDER in MODELS, f"PROVIDER must be 'anthropic' or 'openai', got: {PROVIDER!r}"
MODEL = MODELS[PROVIDER]
print(f"Provider: {PROVIDER}  |  Model: {MODEL}")

## Step 2 — Install dependencies

Both SDKs are installed; only the one you chose will be imported.

In [ ]:
%pip install -q anthropic openai

## Step 3 — API key

Paste your key when prompted. It stays in this session only.

In [ ]:
import os
from getpass import getpass

ENV_VARS = {
    "anthropic": "ANTHROPIC_API_KEY",
    "openai":    "OPENAI_API_KEY",
}
env_var = ENV_VARS[PROVIDER]

os.environ[env_var] = getpass(f"Paste your {PROVIDER.title()} API key ({env_var}): ")
print(f"{env_var} set ✓")

## Step 4 — Build the calculator tool

The agent needs at least one tool. We'll build a **safe calculator** that uses Python's AST parser instead of `eval()`. This matters: the expression comes from the model, which means it's untrusted input.

### Why not `eval(expression)`?

```python
eval("__import__('os').system('rm -rf /')")
```

`eval` would execute that. The AST approach only allows numbers and the four arithmetic operators — anything else returns an error string instead of running code.

### Your TODO

`_safe_eval` is complete. Fill in `calculator()` so it:
1. Parses the expression string into an AST
2. Calls `_safe_eval` on the root node
3. Returns the result as a string
4. Catches any exception and returns a descriptive error string — never crashes

In [ ]:
import ast
import operator as op

_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


def calculator(args: dict) -> str:
    """Safe calculator. args must contain 'expression' key."""
    # TODO: implement
    # Hint: ast.parse(expr, mode="eval").body gives you the root node
    pass


TOOLS = {"calculator": calculator}


# Self-test
assert calculator({"expression": "2 + 2"}) == "4",          "Basic addition failed"
assert calculator({"expression": "10 / 2"}) == "5.0",       "Division failed"
assert "error" in calculator({"expression": "__import__('os')"}).lower(), "Should return error for unsafe input"
print("calculator() tests passed ✓")

<details>
<summary>💡 Stuck? Reveal the calculator solution</summary>

```python
def calculator(args: dict) -> str:
    try:
        tree = ast.parse(args["expression"], mode="eval")
        result = _safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"calculator error: {e}"
```

</details>

## Step 5 — Build the agent loop

The loop:
1. Sends the goal (plus conversation history) to the LLM
2. The LLM replies with JSON — either a tool call or a final answer
3. If a tool call, execute it and append the result
4. Repeat until done or `MAX_STEPS` is reached

### JSON protocol

**Tool call:**
```json
{"thought": "I need to compute 24 × 7", "action": "calculator", "args": {"expression": "24 * 7"}}
```
**Final answer:**
```json
{"thought": "I have the answer", "action": "final", "answer": "168"}
```

### Message format (same for both providers)

Both Anthropic and OpenAI accept a list of `{role, content}` dicts. The system prompt is passed separately.

```
messages = [
  {"role": "user",      "content": "<goal>"},
  {"role": "assistant", "content": "<model reply>"},
  {"role": "user",      "content": "Observation: <tool result>"},
  ...
]
```

### The five TODOs

Fill in the five `pass` statements below. **TODO 2 (`call_model`) is the only place the provider matters.**

In [ ]:
import json
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

MAX_STEPS = 5

SYSTEM_PROMPT = """You are a small tool-using agent. Always reply with valid JSON — nothing else.

To use a tool:
  {"thought": "<your reasoning>", "action": "calculator", "args": {"expression": "<math expression>"}}

To give a final answer:
  {"thought": "<your reasoning>", "action": "final", "answer": "<your answer>"}
"""


def build_initial_messages(goal: str) -> list[dict]:
    """TODO 1: Return [{"role": "user", "content": goal}].
    Both providers share this format. The system prompt is passed separately in call_model.
    """
    # TODO 1
    pass


def call_model(messages: list[dict]) -> str:
    """TODO 2: Call the chosen provider's API and return the model's text response.

    Anthropic:
      import anthropic
      client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
      r = client.messages.create(model=MODEL, max_tokens=256, system=SYSTEM_PROMPT, messages=messages)
      return r.content[0].text

    OpenAI:
      from openai import OpenAI
      client = OpenAI()  # reads OPENAI_API_KEY
      msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
      r = client.chat.completions.create(model=MODEL, max_tokens=256, messages=msgs)
      return r.choices[0].message.content

    Use PROVIDER and MODEL (defined above) to choose the right branch.
    """
    # TODO 2
    pass


def parse_action(text: str) -> dict:
    """TODO 3: Parse the model's JSON response.
    If invalid JSON or missing 'action' key, return a safe default that stops the loop:
      {"action": "final", "answer": "<error description>"}
    """
    # TODO 3
    pass


def run_agent(goal: str) -> str:
    messages = build_initial_messages(goal)

    for step in range(MAX_STEPS):
        raw = call_model(messages)
        action = parse_action(raw)
        log.info("step %d | action: %s", step + 1, action.get("action"))

        # TODO 4: If action["action"] == "final", return action["answer"]

        # TODO 5: Otherwise look up the tool in TOOLS, call it with action["args"],
        #         append the assistant turn and the observation to messages, then continue
        pass

    return "max steps reached — try a simpler goal"

<details>
<summary>💡 Stuck? Reveal the full agent solution</summary>

```python
def build_initial_messages(goal: str) -> list[dict]:
    return [{"role": "user", "content": goal}]


def call_model(messages: list[dict]) -> str:
    if PROVIDER == "anthropic":
        import anthropic
        client = anthropic.Anthropic()
        r = client.messages.create(
            model=MODEL, max_tokens=256,
            system=SYSTEM_PROMPT, messages=messages,
        )
        return r.content[0].text
    elif PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
        r = client.chat.completions.create(
            model=MODEL, max_tokens=256, messages=msgs,
        )
        return r.choices[0].message.content


def parse_action(text: str) -> dict:
    try:
        action = json.loads(text)
    except json.JSONDecodeError:
        return {"action": "final", "answer": "Model returned invalid JSON — stopping safely."}
    if "action" not in action:
        return {"action": "final", "answer": "No action key in model output — stopping safely."}
    return action


def run_agent(goal: str) -> str:
    messages = build_initial_messages(goal)
    for step in range(MAX_STEPS):
        raw = call_model(messages)
        action = parse_action(raw)
        log.info("step %d | action: %s", step + 1, action.get("action"))

        # TODO 4
        if action["action"] == "final":
            return action["answer"]

        # TODO 5
        tool_name = action["action"]
        observation = (
            TOOLS[tool_name](action.get("args", {}))
            if tool_name in TOOLS
            else f"Unknown tool: {tool_name}"
        )
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user",      "content": f"Observation: {observation}"})

    return "max steps reached — try a simpler goal"
```

</details>

## Step 6 — Test your agent

In [ ]:
# Test 1: basic arithmetic — agent should use the calculator tool
result = run_agent("What is 24 multiplied by 7?")
print("Result:", result)
assert "168" in result, f"Expected 168 in result, got: {result}"
print("Test 1 passed ✓")

In [ ]:
# Test 2: multi-step — needs two tool calls
result = run_agent("What is (15 + 27) multiplied by 4?")
print("Result:", result)
assert "168" in result, f"Expected 168 in result, got: {result}"
print("Test 2 passed ✓")

In [ ]:
# Test 3: unknown tool — must not crash
result = run_agent("Search the web for the capital of France.")
print("Result:", result)
print("Test 3 passed ✓ (didn't crash)")

## Step 7 — Experiments

### Experiment A: break the termination
Change `MAX_STEPS = 5` to `MAX_STEPS = 1` and run a multi-step goal. What happens?

### Experiment B: remove the JSON guard
In `parse_action`, remove the `try/except` and replace with `return json.loads(text)`. Run a conversational goal (e.g. "Tell me a joke"). What error do you get? Why does the guard exist?

### Experiment C: temperature
Add `temperature=1.0` to your provider's API call and run the same arithmetic goal five times. Then try `temperature=0.0`. Do you see different behaviour? Why?

### Experiment D: swap the provider
Go back to the provider cell, flip `PROVIDER` to the other value, re-run all cells from there down. Does the agent still pass all three tests? (It should — the loop doesn't change.)

In [ ]:
# Scratch cell — use this for experiments


## Step 8 — Reflection

1. **What does your agent do?**
2. **What breaks if you remove the loop?**
3. **What changes at `temperature=1.0` vs `0.0`?**
4. **What was identical between the two providers, and what was different?**

*(Double-click to edit)*

1. 
2. 
3. 
4. 

## Done!

Head back to the platform and take the **Module Quiz** to mark Module 00 complete.

**Rubric reminder** — Meets (3) on all five criteria:
- Loop runs and terminates correctly
- You can explain every line cold
- You can draw and explain the agent loop correctly
- You can name a problem where a plain LLM call beats an agent loop
- README covers what breaks without the loop and temperature effects